# Experimento — KAN: fase 1, baseline e arquitetura

Este notebook inicia o treinamento da KAN com PyKAN 0.2.8. A execução mede o baseline e percorre blocos de uma variável por vez; nenhum resultado é presumido.

Instale as dependências do projeto (`pip install -r requirements.txt`) e configure `CHURN_CSV` se o CSV não estiver em `dados/telco_ibm.csv`. As próximas fases continuam em `kan_fase2.ipynb` a `kan_fase5.ipynb`.

## Protocolo do experimento

Cada bloco parte do campeão atual e muda uma escolha. A métrica principal é KS médio nas cinco dobras oficiais; AUROC, entropia cruzada e Brier são diagnósticos. O teste não participa de seleção, parada, diagnóstico ou comparação: só será aberto na fase 5, com a configuração congelada.

O scaler e o oversampling são ajustados somente no treino de cada dobra. Empates ficam com a configuração mais simples. Repita as medições com mais sementes antes de interpretar diferenças pequenas.

## Dados

A entrada é a saída de `if702-preprocessamento.ipynb`: as cinco dobras estratificadas e a função `preparar` para o treino final e o teste. Em cada dobra o scaler é ajustado só no treino da dobra.

O oversampling repetitivo já está feito aí. O `RandomOverSampler` só duplica o treino da dobra. A validação e o teste continuam com um cliente por linha. Nenhum bloco refaz esse balanceamento por fora das dobras.

### Estratégias de dados compartilhadas

Nenhuma, além do pré-processamento.

Quando aparecer uma estratégia nova de exploração ou de mineração, ela é escrita nesta seção e copiada para os outros seis notebooks. Vira um bloco em cada experimento, em cima da configuração que aquele modelo já tiver aceito. O bloco mexe só nos dados. O modelo fica parado.


## Frentes já definidas

A versão oficial no PyPI continua a 0.2.8, de 14 de novembro de 2024. Não há release posterior. O treino nessa versão é `fit`. O README ainda escreve `model.train`; esse método não existe no código da tag. `KAN` nessa tag é `MultKAN`.

`fit` com a perda vazia usa erro quadrático. Para o churn, a perda de classificação é escolhida uma vez, antes do bloco 0. Não é um knob da busca.

Cada argumento abaixo é um bloco.

- `width`. Lista com a entrada, as camadas ocultas e a saída. Um inteiro vira só nós de soma. O construtor não tem default de largura. Sem faixa oficial. O README manda encolher `grid` antes de `width` quando treino e teste se afastam.
- `grid`. Número de intervalos da grade do spline. Default 3. Sem faixa oficial.
- `k`. Ordem do B-spline. Default 3. Sem faixa oficial.
- `base_fun`. Default `'silu'`. O construtor só converte `'silu'`, `'identity'` e `'zero'`. Logística, tangente hiperbólica e ReLU não são valores desse argumento. Tanh aparece na biblioteca simbólica, que fica fora desta busca.
- `grid_eps`. Default 0,02, no intervalo documentado de 0 a 1. 1 é grade uniforme e 0 é a partição pelos percentis da amostra.
- `opt`. Default `"LBFGS"`. O outro valor documentado é `"Adam"`. Não há SGD nem RMSprop nesse `fit`.
- `lr`. Default 1,0. Sem faixa oficial. Fica separado do otimizador: a fonte não diz que os dois interagem.
- `steps`. Default 100. Sem faixa oficial.
- `batch`. Default −1, o lote inteiro. Com cerca de 7 mil linhas, o default já é lote cheio.
- `update_grid`. Default `True`. Reposiciona a grade que já existe e não aumenta o número de intervalos. Entram no mesmo bloco os companheiros que só existem para agendar isso: `grid_update_num` (default 10), `start_grid_update_step` (default −1) e `stop_grid_update_step` (default 50).
- `refine(new_grid)`. É um método, não um argumento de `fit`. Sobe o número de intervalos em cima de um modelo já aceito. O exemplo da docstring vai de 5 para 10. O README avisa overfitting.
- Família `lamb`. Com `lamb` em 0, que é o default, o resto da penalidade não entra. `lamb_l1` default 1, `lamb_entropy` default 2, `lamb_coef` default 0 e `lamb_coefdiff` default 0. O README manda começar com `lamb=0`. Não há dropout. A regularização própria da KAN é esta família.
- `scale_base_mu` (default 0) e `scale_base_sigma` (default 1) desenham a magnitude residual inicial. `sb_trainable` e `sp_trainable`, os dois default `True`, decidem se essas magnitudes entram no otimizador. Não existe argumento `scale_spline` no construtor de `MultKAN`.
- Multiplicação. No `width`, o segundo número do par é a quantidade de nós de multiplicação da camada. `mult_arity` default 2, o número de entradas de cada um desses nós. Essa frente só abre depois da KAN só com somas.

Fica de fora o ramo simbólico: `fix_symbolic`, `auto_symbolic`, `prune` e a leitura de fórmula. O README descreve esse ramo para descoberta científica, não para subir a métrica de uma classificação.

O oversampling repetitivo das dobras vale no `fit`.

Fontes: pykan 0.2.8 no PyPI, `kan/MultKAN.py` na tag v0.2.8, e os tutoriais de API 4, 5, 6 e 8 em kindxiaoming.github.io/pykan.


## Fases de treinamento

1. **Este notebook:** baseline e blocos de largura/profundidade, grade, otimizador, passos, lote e funções base; depois, busca Optuna exploratória.
2. `kan_fase2.ipynb`: estratégias de dados e diagnóstico por permutação.
3. `kan_fase3.ipynb`: regularização KAN, escala da função base, refinamento de grade e nós multiplicativos.
4. `kan_fase4.ipynb`: confirmação com separação interna de parada e dobras externas que não decidem os passos.
5. `kan_fase5.ipynb`: placar comum e avaliação final no teste, uma vez.

## Baseline e blocos fatoriais

O baseline segue o roteiro KAN: `width=[n_features, 8, 1]`, `grid=3`, `k=3`, `base_fun='silu'`, LBFGS, `lr=1`, 100 passos, lote cheio, sem penalização e grade adaptativa. A perda é BCE com logits e saída escalar. O seletor abaixo aceita uma variante somente quando o KS médio nas mesmas dobras melhora; veja sempre a dispersão e os ganhos por dobra.

## Interpretação

Os valores exibidos são medidos na execução atual. `MODO_RAPIDO=True` reduz blocos/trials para validar o fluxo e não serve para concluir sobre o modelo. A busca e os blocos rodam sobre validação; não copie pontuações de teste para escolher hiperparâmetros.

## 0. Ambiente e integridade dos dados

In [ ]:
import copy
import json
import os
import sys
from pathlib import Path

CSV_PATH = os.environ.get("CHURN_CSV", "")
if CSV_PATH:
    os.environ["CHURN_CSV"] = CSV_PATH
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "comum.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Abra o notebook dentro do repositório churn_project.")
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import comum
from experimentos.kan import modelo as K

RESULTADOS = Path(comum.RESULTADOS)
PASTA_KAN = RESULTADOS / "kan"
PASTA_KAN.mkdir(parents=True, exist_ok=True)
print(f"Repositório: {ROOT}")
print(f"Dispositivo: {K.DEVICE}")
print(f"CSV: {comum.CAMINHO_CSV}")

In [ ]:
MODO_RAPIDO = False
SEMENTES = (42,) if MODO_RAPIDO else (42, 142, 242)
N_TRIALS = 2 if MODO_RAPIDO else 100
folds = K.carregar_dobras()
assert len(folds) == 5
assert all(len(f) == 4 for f in folds)
print("Dobras:", len(folds), "| dimensões:", folds[0][0].shape[1],
      "| treino da primeira dobra:", folds[0][0].shape, "| validação:", folds[0][2].shape)

## 1. Referência e comparação pareada

In [ ]:
campeao = copy.deepcopy(K.KAN_PADRAO)
medida_ref = K.medir_config(campeao, folds, SEMENTES)
historico = [{
    "bloco": "baseline",
    "ks": medida_ref["ks"],
    "ks_dp": medida_ref["ks_dp"],
    "config": copy.deepcopy(campeao),
}]
display(pd.DataFrame(medida_ref["linhas"]))

In [ ]:
BLOCOS = [
    ("largura", lambda base: [("4", {"hidden": (4,)}), ("16", {"hidden": (16,)})]),
    ("profundidade", lambda base: [("duas camadas", {"hidden": (base["hidden"][0], base["hidden"][0])})]),
    ("grid", lambda base: [("grid=5", {"grid": 5}), ("grid=7", {"grid": 7})]),
    ("ordem k", lambda base: [("k=2", {"k": 2})]),
    ("otimizador", lambda base: [("Adam", {"opt": "Adam"})]),
    ("taxa de aprendizagem", lambda base: [("lr=0.3", {"lr": 0.3}), ("lr=1.5", {"lr": 1.5})]),
    ("passos", lambda base: [("50", {"steps": 50}), ("150", {"steps": 150})]),
    ("lote", lambda base: [("256", {"batch": 256}), ("512", {"batch": 512})]),
    ("função base", lambda base: [("identity", {"base_fun": "identity"}), ("zero", {"base_fun": "zero"})]),
    ("grid_eps", lambda base: [("0.2", {"grid_eps": 0.2}), ("0.5", {"grid_eps": 0.5}), ("1.0", {"grid_eps": 1.0})]),
    ("atualização da grade", lambda base: [
        ("grade fixa", {"update_grid": False}),
        ("5 atualizações", {"grid_update_num": 5}),
    ]),
]

for i, (nome, criar_opcoes) in enumerate(BLOCOS):
    if MODO_RAPIDO and i >= 2:
        break
    opcoes = criar_opcoes(campeao)
    variantes = {rotulo: K.refinar_hp(campeao, **mudanca) for rotulo, mudanca in opcoes}
    referencia, resultados = K.comparar_bloco(campeao, variantes, folds, SEMENTES)
    tabela = pd.DataFrame(resultados).sort_values(["ks", "ganhou_dobras"], ascending=False)
    display(tabela[["variante", "status", "ks", "ks_dp", "delta_ks", "ganhou_dobras",
                    "empatou_dobras", "perdeu_dobras", "erro"]])
    validos = tabela[tabela["status"] == "ok"]
    if validos.empty:
        ks_escolhido, ks_dp_escolhido = referencia["ks"], referencia["ks_dp"]
        decisao = "mantido: nenhum candidato válido"
    else:
        melhor = validos.iloc[0]
        ks_escolhido = max(referencia["ks"], melhor["ks"])
        ks_dp_escolhido = melhor["ks_dp"]
        if melhor["delta_ks"] > 0:
            campeao = copy.deepcopy(melhor["config"])
            decisao = str(melhor["variante"])
        else:
            decisao = "mantido: nenhum candidato superou a referência"
    historico.append({
        "bloco": nome, "decisao": decisao, "ks": float(ks_escolhido),
        "ks_dp": float(ks_dp_escolhido), "config": copy.deepcopy(campeao),
        "falhas": [{"variante": r["variante"], "erro": r["erro"]}
                   for r in resultados if r["status"] == "falhou"],
    })

display(pd.DataFrame([{k: r[k] for k in ("bloco", "decisao", "ks", "ks_dp")} for r in historico]))

## 2. Busca Optuna exploratória

In [ ]:
# A busca é exploratória, não substitui os blocos fatoriais nem a confirmação da fase 4.
estudo = comum.estudo("kan", N_TRIALS, K.hp_kan, K.fit, K.KAN_TRIAL0)
hp_optuna = K.hp_from_params(estudo.best_params)
medida_optuna = K.medir_config(hp_optuna, folds, SEMENTES)
print("Melhor KS Optuna:", medida_optuna["ks"])
display(pd.DataFrame(estudo.trials_dataframe()).tail(10))

if medida_optuna["ks"] > K.medir_config(campeao, folds, SEMENTES)["ks"]:
    campeao = hp_optuna
    origem = "optuna"
else:
    origem = "blocos fatoriais"

saida = {"config": json_config(campeao), "origem": origem, "historico": historico,
         "optuna_best_params": estudo.best_params, "modo_rapido": MODO_RAPIDO}
(PASTA_KAN / "kan_fase1.json").write_text(json.dumps(saida, ensure_ascii=False, indent=2), encoding="utf-8")
print("Campeão provisório:", origem, campeao)
print("Salvo em:", PASTA_KAN / "kan_fase1.json")